In [28]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline 

In [6]:
df = pd.read_csv('amz_pdp_price_sales_daily.csv')

In [8]:
df

,crawl_date,asin,bw_price,ordered_revenue,ordered_units
0,2025-06-08,B0CKYSMM1J,118.78,9264.84,78.0
1,2025-06-08,B0CKYYL8B9,181.07,30143.84,166.0
2,2025-06-08,B0CKYZCVXK,100.99,31100.92,308.0
3,2025-06-08,B0CSJTDWXS,141.31,5214.37,37.0
4,2025-06-08,B0CSK2VDKG,254.64,27246.48,107.0
...,...,...,...,...,...
563181,2023-08-17,B09HZKT3X1,199.00,3184.00,16.0
563182,2023-08-17,B072QBWG7W,351.99,5631.84,16.0
563183,2023-08-17,B017YETH8K,97.30,1556.80,16.0
563184,2023-08-17,B095NYDG9X,89.00,1424.00,16.0


In [10]:
df1 = df[(df['ordered_units'] > 0) & (df['bw_price'] > 0)].copy()

In [12]:
df1

,crawl_date,asin,bw_price,ordered_revenue,ordered_units
0,2025-06-08,B0CKYSMM1J,118.78,9264.84,78.0
1,2025-06-08,B0CKYYL8B9,181.07,30143.84,166.0
2,2025-06-08,B0CKYZCVXK,100.99,31100.92,308.0
3,2025-06-08,B0CSJTDWXS,141.31,5214.37,37.0
4,2025-06-08,B0CSK2VDKG,254.64,27246.48,107.0
...,...,...,...,...,...
563181,2023-08-17,B09HZKT3X1,199.00,3184.00,16.0
563182,2023-08-17,B072QBWG7W,351.99,5631.84,16.0
563183,2023-08-17,B017YETH8K,97.30,1556.80,16.0
563184,2023-08-17,B095NYDG9X,89.00,1424.00,16.0


In [14]:
df1['log_units'] = np.log(df1['ordered_units'])
df1['log_price'] = np.log(df1['bw_price'])

In [16]:
# 2) inf, -inf → NaN 으로 변환
df1.replace([np.inf, -np.inf], np.nan, inplace=True)

# 3) 로그 변수에 NaN 혹은 결측치가 있는 행 제거
df1.dropna(subset=['log_units', 'log_price'], inplace=True)

In [18]:
df1

,crawl_date,asin,bw_price,ordered_revenue,ordered_units,log_units,log_price
0,2025-06-08,B0CKYSMM1J,118.78,9264.84,78.0,4.356709,4.777273
1,2025-06-08,B0CKYYL8B9,181.07,30143.84,166.0,5.111988,5.198884
2,2025-06-08,B0CKYZCVXK,100.99,31100.92,308.0,5.730100,4.615022
3,2025-06-08,B0CSJTDWXS,141.31,5214.37,37.0,3.610918,4.950956
4,2025-06-08,B0CSK2VDKG,254.64,27246.48,107.0,4.672829,5.539851
...,...,...,...,...,...,...,...
563181,2023-08-17,B09HZKT3X1,199.00,3184.00,16.0,2.772589,5.293305
563182,2023-08-17,B072QBWG7W,351.99,5631.84,16.0,2.772589,5.863603
563183,2023-08-17,B017YETH8K,97.30,1556.80,16.0,2.772589,4.577799
563184,2023-08-17,B095NYDG9X,89.00,1424.00,16.0,2.772589,4.488636


In [22]:
X = df1[['log_price']].values
y = df1['log_units'].values

In [24]:
# 1) 훈련/테스트 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [30]:

# 2) 비교할 모델 정의
models = {
    'Linear': LinearRegression(),
    'Poly2': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('lr',   LinearRegression())
    ]),
    'RF': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGB': XGBRegressor(n_estimators=100, max_depth=3,
                        learning_rate=0.1, random_state=42)
}


In [32]:
# (4) 학습·예측·평가
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results.append({
        'model': name,
        'R2':    r2_score(y_test, y_pred),
        'RMSE':  np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE':   mean_absolute_error(y_test, y_pred)
    })
metrics_df = pd.DataFrame(results).set_index('model')
print(metrics_df)

              R2      RMSE       MAE
model                               
Linear  0.004661  1.256291  1.014142
Poly2   0.005596  1.255701  1.013263
RF      0.259112  1.083880  0.827985
XGB     0.045801  1.230054  0.993336
